# Task 0: Colab environment and baseline render

Choose **Runtime → Change runtime type → GPU**, then run the notebook from top to bottom. The first cell clones or fast-forwards the homework repository from GitHub, so push local changes to the repository before rerunning it in Colab. The notebook checks the CUDA runtime and compiler, installs project dependencies and builds the rasterizer, then renders and packages the starter output. The build may take several minutes.

If the GPU or `nvcc` check fails, stop and select a GPU runtime; do not continue with a CPU runtime.

In [3]:
import os
import pathlib
import subprocess

repo_url = 'https://github.com/jon-of-us/gaussian-splatting-homwork.git'
assignment_root = pathlib.Path('/content/hw1_2026_student')
if (assignment_root / '.git').is_dir():
    subprocess.run(['git', '-C', str(assignment_root), 'pull', '--ff-only'], check=True)
elif assignment_root.exists():
    raise FileExistsError(
        f'{assignment_root} exists but is not a Git checkout. Rename or remove that folder in Colab, then rerun this cell.'
    )
else:
    subprocess.run(['git', 'clone', '--depth', '1', repo_url, str(assignment_root)], check=True)

if not (assignment_root / 'render.py').is_file() or not (assignment_root / 'scripts/setup_environment.sh').is_file():
    raise FileNotFoundError(f'Homework repository is missing required files: {assignment_root}')
os.chdir(assignment_root)
print(f'Assignment directory: {pathlib.Path.cwd()}')

Cloning assignment repository into /content/tsinghua


CalledProcessError: Command '['git', 'clone', '--depth', '1', 'https://github.com/jon-of-us/tsinghua.git', '/content/tsinghua']' returned non-zero exit status 128.

In [ ]:
import torch
import torch.utils.cpp_extension
import torchvision

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable. Select a GPU under Runtime → Change runtime type, then rerun this notebook.')
if torch.utils.cpp_extension.CUDA_HOME is None:
    raise RuntimeError('CUDA Toolkit was not found. Select a Colab GPU runtime with nvcc available.')

subprocess.run(['nvidia-smi'], check=True)
subprocess.run(['nvcc', '--version'], check=True)
print(f'Python: {os.sys.version.split()[0]}')
print(f'PyTorch: {torch.__version__}')
print(f'torchvision: {torchvision.__version__}')
print(f'PyTorch CUDA: {torch.version.cuda}')
print(f'GPU: {torch.cuda.get_device_name(0)}')

## Install dependencies and build the CUDA extension

The repository helper uses the CUDA-enabled PyTorch already installed in Colab. It checks compatibility, installs `requirements.txt`, runs the runtime interoperability checks, and compiles the rasterization extension. The job limit is set to two to reduce peak memory use.

In [ ]:
os.environ['MAX_JOBS'] = '2'
subprocess.run(['bash', 'scripts/setup_environment.sh'], check=True)
import importlib

rasterizer_module = importlib.import_module('diff_gaussian_rasterization')
cuda_extension = importlib.import_module('diff_gaussian_rasterization._C')
print(f'Rasterizer: {rasterizer_module.__file__}')
print(f'CUDA extension: {cuda_extension.__file__}')

## Render and preserve the starter baseline

This runs the starter implementation before any TODOs are changed. It copies all five generated views to a separate directory so later runs cannot overwrite the baseline, displays view 0, and downloads a ZIP containing the baseline images and environment details. Extract the archive into the local `report/` directory.

In [ ]:
import shutil

subprocess.run([os.sys.executable, 'render.py'], check=True)
render_dir = pathlib.Path('renders')
baseline_dir = pathlib.Path('report/task0_baseline_views')
baseline_dir.mkdir(parents=True, exist_ok=True)

baseline_images = [render_dir / f'{index:05d}.png' for index in range(5)]
missing_images = [str(path) for path in baseline_images if not path.is_file()]
if missing_images:
    raise FileNotFoundError(f'Expected five rendered camera views; missing: {missing_images}')
for image_path in baseline_images:
    shutil.copy2(image_path, baseline_dir / image_path.name)

environment_report = pathlib.Path('report/task0_environment.txt')
environment_report.write_text(
    '\n'.join([
        f'Python: {os.sys.version.split()[0]}',
        f'PyTorch: {torch.__version__}',
        f'torchvision: {torchvision.__version__}',
        f'PyTorch CUDA: {torch.version.cuda}',
        f'GPU: {torch.cuda.get_device_name(0)}',
        f'CUDA Toolkit: {torch.utils.cpp_extension.CUDA_HOME}',
    ]) + '\n',
    encoding='utf-8',
)
print(f'Preserved {len(baseline_images)} baseline images in {baseline_dir}')
print(environment_report.read_text(encoding='utf-8'))

In [ ]:
import google.colab.files
import IPython.display
import zipfile

IPython.display.display(IPython.display.Image(filename=str(baseline_dir / '00000.png')))
archive_path = pathlib.Path('/content/task0_artifacts.zip')
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for image_path in sorted(baseline_dir.glob('*.png')):
        archive.write(image_path, arcname=f'task0_baseline_views/{image_path.name}')
    archive.write(environment_report, arcname='task0_environment.txt')
google.colab.files.download(str(archive_path))